In [1]:
#!/usr/bin/env python3
"""
WR / LRSM Analysis  ─  uproot + awkward-array + coffea 기반 구현
원본: Reproduce20_002_copy.cpp (SKNano 프레임워크) 를 NanoAOD 기준으로 번역

물리 목표: WR → l + N_R → l + l + jj  (Left-Right Symmetric Model)

두 가지 위상학적 선택:
  Resolved : tight lepton 2개 + AK4 jet 2개 이상  →  m(lljj)
  Boosted  : tight lepton 1개 + AK8 fat-jet 1개   →  m(l + FJ)

실행 방법:
  pip install uproot awkward coffea hist vector
  python wr_analysis_uproot.py --input sample.root --era 2022 [--is-data]
"""

from __future__ import annotations

import argparse
import warnings
from typing import Dict, List

import awkward as ak
import hist
import numpy as np
import uproot
import vector

# coffea NanoAOD schema (4-벡터 동작 자동 부여)
from coffea.nanoevents import NanoAODSchema, NanoEventsFactory

vector.register_awkward()   # ak.Array 에 .mass, .delta_r 등 활성화

# ══════════════════════════════════════════════════════════════
#  ERA별 트리거 / 안전 pT 기준 (initializeAnalyzer 대응)
# ══════════════════════════════════════════════════════════════

ERA_CFG: Dict[str, Dict] = {
    "2017": dict(
        mu_trig = ["HLT_Mu50", "HLT_OldMu100", "HLT_TkMu100"],
        mu_pt   = 52.0,
        el_trig = ["HLT_Ele35_WPTight_Gsf",
                   "HLT_Photon200",
                   "HLT_Ele115_CaloIdVT_GsfTrkIdT"],
        el_pt   = 38.0,
    ),
    "2022": dict(
        mu_trig = ["HLT_Mu50", "HLT_CascadeMu100", "HLT_HighPtTkMu100"],
        mu_pt   = 52.0,
        el_trig = ["HLT_Photon200", "HLT_Ele115_CaloIdVT_GsfTrkIdT"],
        el_pt   = 118.0,
    ),
    "2022EE": dict(
        mu_trig = ["HLT_Mu50", "HLT_CascadeMu100", "HLT_HighPtTkMu100"],
        mu_pt   = 52.0,
        el_trig = ["HLT_Photon200", "HLT_Ele115_CaloIdVT_GsfTrkIdT"],
        el_pt   = 118.0,
    ),
    "2023": dict(
        mu_trig = ["HLT_Mu50", "HLT_CascadeMu100", "HLT_HighPtTkMu100"],
        mu_pt   = 52.0,
        el_trig = ["HLT_Photon200", "HLT_Ele115_CaloIdVT_GsfTrkIdT"],
        el_pt   = 118.0,
    ),
    "2023BPix": dict(
        mu_trig = ["HLT_Mu50", "HLT_CascadeMu100", "HLT_HighPtTkMu100"],
        mu_pt   = 52.0,
        el_trig = ["HLT_Photon200", "HLT_Ele115_CaloIdVT_GsfTrkIdT"],
        el_pt   = 118.0,
    ),
}

# 객체 선택 임계값
EL_MIN_PT    = 10.0;  EL_MAX_ETA  = 2.5
MU_MIN_PT    = 10.0;  MU_MAX_ETA  = 2.4
JET_MIN_PT   = 30.0;  JET_MAX_ETA = 2.4
FJ_MIN_PT    = 200.0; FJ_MAX_ETA  = 2.4
FJ_MIN_SDM   = 10.0   # soft-drop mass 최솟값 [GeV]
FJ_LSF3_CUT  = 0.3    # Lepton Sub-jet Fraction 임계값


# ══════════════════════════════════════════════════════════════
#  전자 ID 함수들  (isPassLooseNoIso / CustomLoose / CustomTight)
# ══════════════════════════════════════════════════════════════

def el_pass_loose_no_iso(bitmap: ak.Array) -> ak.Array:
    """
    VidNestedWPBitmap 에서 격리(isolation)를 제외한 Loose WP 합격 여부.

    NanoAOD branch : Electron_vidNestedWPBitmap

    비트 구조 (cut_nr × 3비트, LSB 기준):
      0 MinPtCut          1 SCEtaMultiRange   2 DEtaInSeed
      3 DPhiIn            4 Full5x5SiEiE      5 HoverEScaled
      6 EInvMinusPInv     7 RelPFIsoScaled ← SKIP
      8 ConversionVeto    9 MissingHits
    3비트 값: 0=fail 1=Veto 2=Loose 3=Medium 4=Tight
    """
    SKIP   = 7       # 격리 항목 번호
    LEVEL  = 2       # Loose 수준
    MASK3  = 0b111   # 3비트 마스크

    result = ak.ones_like(bitmap, dtype=bool)
    for cut_nr in range(10):
        if cut_nr == SKIP:
            continue
        val    = (bitmap >> (cut_nr * 3)) & MASK3
        result = result & (val >= LEVEL)
    return result


def el_custom_loose_id(
    sceta    : ak.Array,   # Electron_superclusterEta  (또는 Electron_eta 근사)
    hoe      : ak.Array,   # Electron_hoe
    sieie    : ak.Array,   # Electron_sieie
    deta_seed: ak.Array,   # Electron_deltaEtaInSeed
    dphi_sc  : ak.Array,   # Electron_deltaPhiInSC  (NanoAOD: Electron_deltaPhiInSC 또는 deltaPhiSuperClusterTrackAtVtx)
    einvpinv : ak.Array,   # Electron_eInvMinusPInv
    lost_hits: ak.Array,   # Electron_lostHits
    conv_veto: ak.Array,   # Electron_convVeto
    energy   : ak.Array,   # Electron_energy  (= pt * cosh(eta))
    rho      : ak.Array,   # fixedGridRhoFastjetAll  (이벤트 수준)
) -> ak.Array:
    """
    커스텀 느슨한(Loose) 전자 ID - 격리 없음.
    Barrel: |scEta| ≤ 1.479   /   Endcap: |scEta| > 1.479
    """
    is_barrel = abs(sceta) <= 1.479

    barrel = (
        (hoe   < 0.05 + 1.16 / energy + 0.0324 * rho / energy) &
        (sieie < 0.0112) &
        (abs(deta_seed) < 0.00377) &
        (abs(dphi_sc)   < 0.0884)  &
        (abs(einvpinv)  < 0.193)   &
        (lost_hits <= 1) & conv_veto
    )
    endcap = (
        (hoe   < 0.0441 + 2.54 / energy + 0.183 * rho / energy) &
        (sieie < 0.0425) &
        (abs(deta_seed) < 0.00674) &
        (abs(dphi_sc)   < 0.169)  &
        (abs(einvpinv)  < 0.111)  &
        (lost_hits <= 1) & conv_veto
    )
    return ak.where(is_barrel, barrel, endcap)


def el_custom_tight_id(
    sceta    : ak.Array,   # Electron_superclusterEta
    cutbased : ak.Array,   # Electron_cutBased >= 4  (Tight WP Boolean)
    heepbit  : ak.Array,   # Electron_vidNestedWPBitmapHEEP
    hoe      : ak.Array,   # Electron_hoe
    pt       : ak.Array,   # Electron_pt  (scEt 근사 - 정확히는 scEt = pt*cosh(scEta)/cosh(eta))
    rho      : ak.Array,   # fixedGridRhoFastjetAll
    dr03ecal : ak.Array,   # Electron_dr03EcalRecHitSumEt
    dr03hcal : ak.Array,   # Electron_dr03HcalDepth1TowerSumEt
) -> ak.Array:
    """
    커스텀 강한(Tight) 전자 ID.
      Barrel |scEta| < 1.566  → CutBased Tight
      Endcap |scEta| > 1.566  → 커스텀 HEEP 기반 컷
    """
    in_barrel = abs(sceta) < 1.566

    # Endcap: HEEP 기반 컷  ─────────────────────────────────
    # 주의: 원 코드에서 scE = (scEtOverPt + 1) * pt 이나,
    #        여기서는 pt 를 scEt 근사로 사용 (정확한 scEt 브랜치가 있으면 교체)
    scE = pt

    hoe_cut    = (-0.4 + 0.4 * abs(sceta)) * rho / scE + 0.05
    emhad_cut  = ak.where(
        pt > 50.0,
        2.5 + 0.03 * (pt - 50.0) + (0.15 + 0.07 * abs(sceta)) * rho,
        2.5                       + (0.15 + 0.07 * abs(sceta)) * rho,
    )

    # 3775 = 0b111010111111 : HEEP 비트마스크
    endcap = (
        ((heepbit & 3775) == 3775) &
        (hoe < hoe_cut) &
        (dr03ecal + dr03hcal < emhad_cut)
    )

    return ak.where(in_barrel, cutbased, endcap)


# ══════════════════════════════════════════════════════════════
#  기하학 보조 함수
# ══════════════════════════════════════════════════════════════

def delta_phi_abs(phi1: ak.Array, phi2: ak.Array) -> ak.Array:
    """|Δφ| ∈ [0, π]"""
    return abs(np.arctan2(np.sin(phi1 - phi2), np.cos(phi1 - phi2)))


def delta_r2(eta1, phi1, eta2, phi2):
    """ΔR² (스칼라 또는 배열)"""
    dphi = np.arctan2(np.sin(phi1 - phi2), np.cos(phi1 - phi2))
    return (eta1 - eta2)**2 + dphi**2


def dr_clean(obj_eta, obj_phi, ref_eta, ref_phi, dr_cut: float = 0.4) -> ak.Array:
    """
    ΔR 기반 객체 정리.
    반환: 객체 마스크 (True = 보존 – 모든 ref 와 dR > dr_cut)

    obj_eta/phi : shape [event, n_obj]
    ref_eta/phi : shape [event, n_ref]
    """
    if ak.all(ak.num(ref_eta) == 0):   # ref가 없으면 전부 보존
        return ak.ones_like(obj_eta, dtype=bool)

    deta = obj_eta[:, :, None] - ref_eta[:, None, :]
    dphi = np.arctan2(
        np.sin(obj_phi[:, :, None] - ref_phi[:, None, :]),
        np.cos(obj_phi[:, :, None] - ref_phi[:, None, :]),
    )
    dr2  = deta**2 + dphi**2
    return ak.all(dr2 > dr_cut**2, axis=2)


# ══════════════════════════════════════════════════════════════
#  트리거 판단
# ══════════════════════════════════════════════════════════════

def pass_trigger(events: ak.Array, trig_list: List[str]) -> ak.Array:
    """트리거 목록의 OR 결과 반환"""
    result = ak.zeros_like(events.run, dtype=bool)
    for name in trig_list:
        try:
            result = result | getattr(events.HLT, name)
        except AttributeError:
            warnings.warn(f"트리거 '{name}' 를 트리에서 찾을 수 없음 – 건너뜀")
    return result


# ══════════════════════════════════════════════════════════════
#  히스토그램 생성
# ══════════════════════════════════════════════════════════════

def book_histograms() -> Dict[str, hist.Hist]:
    h = {}

    def add(name: str, nbins: int, lo: float, hi: float, label: str = ""):
        h[name] = hist.Hist(
            hist.axis.Regular(nbins, lo, hi, name="x", label=label),
            hist.axis.StrCategory([], name="region", growth=True),
            storage=hist.storage.Weight(),
        )

    # ── Resolved ────────────────────────────────────────────
    add("Resolve_mll",   100,    0, 2000, r"$m_{ll}$ [GeV]")
    add("Resolve_mlljj", 200,    0, 8000, r"$m_{lljj}$ [GeV]")
    add("Resolve_l1pt",  100,    0, 2000, r"Lead lepton $p_T$ [GeV]")
    add("Resolve_l2pt",  100,    0, 1000, r"Sublead lepton $p_T$ [GeV]")
    add("Resolve_j1pt",  100,    0, 2000, r"Lead jet $p_T$ [GeV]")
    add("Resolve_j2pt",  100,    0, 1000, r"Sublead jet $p_T$ [GeV]")
    add("Resolve_njet",   20,    0,   20, r"$N_\mathrm{jets}$")

    # ── Boosted ─────────────────────────────────────────────
    add("Boost_mlljj",   200,    0, 8000, r"$m_{l+\mathrm{FJ}}$ [GeV]")
    add("Boost_mll",     100,    0, 2000, r"$m_{ll}$ (DY CR) [GeV]")
    add("Boost_l1pt",    100,    0, 2000, r"Lead lepton $p_T$ [GeV]")
    add("Boost_fjpt",    100,    0, 2000, r"Fat jet $p_T$ [GeV]")
    add("Boost_fjsdm",   100,    0,  500, r"Fat jet $m_\mathrm{SD}$ [GeV]")
    add("Boost_fjlsf3",  100,    0,    1, r"LSF3")
    add("Boost_dphi",    100,    0,  3.5, r"$|\Delta\phi(l, \mathrm{FJ})|$")

    return h


def fill_h(h: Dict, name: str, region: str,
           vals: ak.Array, wgt: ak.Array) -> None:
    """히스토그램 채우기 (빈 배열 안전 처리)"""
    if ak.any(ak.num(vals) > 0 if vals.ndim > 1 else len(vals) > 0):
        v = ak.to_numpy(ak.flatten(vals, axis=None))
        w = ak.to_numpy(ak.flatten(wgt,  axis=None))
        if len(v) > 0:
            h[name].fill(x=v, region=region, weight=w)


# ══════════════════════════════════════════════════════════════
#  메인 이벤트 분석  (executeEventFromParameter 대응)
# ══════════════════════════════════════════════════════════════

def analyze_events(events: ak.Array, era: str,
                   is_data: bool, h: Dict[str, hist.Hist]) -> None:
    cfg = ERA_CFG[era]
    rho = events.fixedGridRhoFastjetAll   # NanoAOD: 이벤트당 에너지 밀도

    # ── 가중치 (MC 전용) ─────────────────────────────────────
    weight = ak.ones_like(events.run, dtype=float)
    if not is_data:
        weight = weight * events.genWeight   # MC 생성 가중치

    # ── 트리거 판단 ──────────────────────────────────────────
    trig_el = pass_trigger(events, cfg["el_trig"])
    trig_mu = pass_trigger(events, cfg["mu_trig"])

    # ════════════════════════════════════════════════════════
    #  전자 선택
    #  NanoAOD 브랜치 참고:
    #    Electron_pt, Electron_eta, Electron_phi, Electron_mass
    #    Electron_superclusterEta  (또는 Electron_scEta – NanoAOD v9+)
    #    Electron_hoe, Electron_sieie
    #    Electron_deltaEtaInSeed, Electron_deltaPhiInSC
    #    Electron_eInvMinusPInv
    #    Electron_lostHits, Electron_convVeto
    #    Electron_vidNestedWPBitmap, Electron_vidNestedWPBitmapHEEP
    #    Electron_cutBased  (0=fail 1=Veto 2=Loose 3=Medium 4=Tight)
    #    Electron_dr03EcalRecHitSumEt, Electron_dr03HcalDepth1TowerSumEt
    # ════════════════════════════════════════════════════════
    el = events.Electron
    el = el[(el.pt > EL_MIN_PT) & (abs(el.eta) < EL_MAX_ETA)]
    el = el[ak.argsort(el.pt, axis=1, ascending=False)]

    # scEta: Electron_superclusterEta 없으면 eta로 근사
    try:
        sceta = el.superclusterEta
    except AttributeError:
        sceta = el.eta   # 근사; 정확한 결과를 원하면 브랜치 명 확인

    el_energy = el.pt * np.cosh(el.eta)   # E ≈ |p| (경전적 근사)
    rho_bc    = rho * ak.ones_like(el.pt) # 이벤트 스칼라를 객체 배열로 브로드캐스트

    # 강한(Tight) 전자 마스크
    tight_el_mask = el_custom_tight_id(
        sceta    = sceta,
        cutbased = el.cutBased >= 4,
        heepbit  = el.vidNestedWPBitmapHEEP,
        hoe      = el.hoe,
        pt       = el.pt,
        rho      = rho_bc,
        dr03ecal = el.dr03EcalRecHitSumEt,
        dr03hcal = el.dr03HcalDepth1TowerSumEt,
    )

    # 느슨한(Loose) 전자 마스크: LooseNoIso OR HEEP 합격
    try:
        heep_pass = el.cutBased_HEEP   # NanoAOD에 있을 경우
    except AttributeError:
        heep_pass = ((el.vidNestedWPBitmapHEEP & 3775) == 3775)

    loose_el_mask = el_pass_loose_no_iso(el.vidNestedWPBitmap) | heep_pass

    tight_el = el[tight_el_mask]
    loose_el = el[loose_el_mask]

    # ════════════════════════════════════════════════════════
    #  뮤온 선택
    #  NanoAOD 브랜치:
    #    Muon_pt, Muon_eta, Muon_phi, Muon_mass
    #    Muon_highPtId   (0=fail 1=Tracker 2=GlobalHighPt)
    #    Muon_tkRelIso   (트래커 상대 격리)
    #    Muon_looseId    (Loose WP boolean)
    #    Muon_tightCharge (0,1,2 – 2 = tight)
    # ════════════════════════════════════════════════════════
    mu = events.Muon
    mu = mu[(mu.pt > MU_MIN_PT) & (abs(mu.eta) < MU_MAX_ETA)]
    mu = mu[ak.argsort(mu.pt, axis=1, ascending=False)]

    # 강한(Tight) 뮤온: GlobalHighPt ID + TkRelIso < 0.1
    tight_mu_mask = (mu.highPtId >= 2) & (mu.tkRelIso < 0.1)
    # 느슨한(Loose) 뮤온: POG Loose
    loose_mu_mask = mu.looseId

    tight_mu = mu[tight_mu_mask]
    loose_mu = mu[loose_mu_mask]

    # ════════════════════════════════════════════════════════
    #  AK4 제트 선택 및 정리
    #  NanoAOD 브랜치: Jet_pt, Jet_eta, Jet_phi, Jet_mass, Jet_jetId
    # ════════════════════════════════════════════════════════
    jet = events.Jet
    jet = jet[
        (jet.pt > JET_MIN_PT) & (abs(jet.eta) < JET_MAX_ETA) & (jet.jetId >= 2)
    ]
    # 느슨한 렙톤과 ΔR < 0.4 이면 제거 (Clean_jet_with_loose_leptons)
    loose_eta = ak.concatenate([loose_el.eta, loose_mu.eta], axis=1)
    loose_phi = ak.concatenate([loose_el.phi, loose_mu.phi], axis=1)
    jet = jet[dr_clean(jet.eta, jet.phi, loose_eta, loose_phi, 0.4)]
    jet = jet[ak.argsort(jet.pt, axis=1, ascending=False)]

    # ════════════════════════════════════════════════════════
    #  AK8 Fat Jet 선택 및 정리
    #  NanoAOD 브랜치: FatJet_pt, _eta, _phi, _mass
    #    FatJet_msoftdrop  (soft-drop mass)
    #    FatJet_lsf3       (Lepton Sub-jet Fraction 3)
    #    FatJet_jetId
    # ════════════════════════════════════════════════════════
    fj = events.FatJet
    fj = fj[
        (fj.pt > FJ_MIN_PT) & (abs(fj.eta) < FJ_MAX_ETA) &
        (fj.msoftdrop > FJ_MIN_SDM) & (fj.jetId >= 2)
    ]
    # 강한 렙톤과 ΔR < 0.4 이면 제거 (Clean_Fatjet_with_tight_leptons)
    tight_eta = ak.concatenate([tight_el.eta, tight_mu.eta], axis=1)
    tight_phi = ak.concatenate([tight_el.phi, tight_mu.phi], axis=1)
    fj = fj[dr_clean(fj.eta, fj.phi, tight_eta, tight_phi, 0.4)]
    fj = fj[ak.argsort(fj.pt, axis=1, ascending=False)]

    # LSF3 fat jet (부스트된 신호 선택용)
    fj_lsf = fj[fj.lsf3 > FJ_LSF3_CUT]
    fj_lsf = fj_lsf[ak.argsort(fj_lsf.pt, axis=1, ascending=False)]

    # ── 렙톤 수 ─────────────────────────────────────────────
    n_tight = ak.num(tight_el) + ak.num(tight_mu)
    is_EE   = (ak.num(tight_el) == 2) & (ak.num(tight_mu) == 0)
    is_MM   = (ak.num(tight_mu) == 2) & (ak.num(tight_el) == 0)
    is_EM   = (ak.num(tight_el) == 1) & (ak.num(tight_mu) == 1)

    # ════════════════════════════════════════════════════════
    #  RESOLVED 선택 (IsResolvedEvent)
    # ════════════════════════════════════════════════════════
    # 조건: tight lepton 정확히 2개, pT > 60/53, 제트 ≥ 2, ΔR 컷
    has2j = ak.num(jet) >= 2

    def jet_dr_pass(l1_eta, l1_phi, l2_eta, l2_phi) -> ak.Array:
        """ΔR > 0.4: j1-l1, j1-l2, j2-l1, j2-l2, j1-j2, l1-l2"""
        j1e, j1p = jet[:, 0].eta, jet[:, 0].phi
        j2e, j2p = jet[:, 1].eta, jet[:, 1].phi
        ok = (
            (delta_r2(j1e, j1p, l1_eta, l1_phi) > 0.16) &
            (delta_r2(j1e, j1p, l2_eta, l2_phi) > 0.16) &
            (delta_r2(j2e, j2p, l1_eta, l1_phi) > 0.16) &
            (delta_r2(j2e, j2p, l2_eta, l2_phi) > 0.16) &
            (delta_r2(j1e, j1p, j2e, j2p)       > 0.16) &
            (delta_r2(l1_eta, l1_phi, l2_eta, l2_phi) > 0.16)
        )
        return ok

    # ── EE Resolved ─────────────────────────────────────────
    ev_res_EE_base = (
        is_EE & (n_tight == 2) & has2j &
        (tight_el[:, 0].pt > 60.0) & (tight_el[:, 1].pt > 53.0) &
        (tight_el[:, 0].pt > cfg["el_pt"]) & trig_el
    )
    ev_res_EE_dr = ak.where(
        ev_res_EE_base,
        jet_dr_pass(tight_el[:, 0].eta, tight_el[:, 0].phi,
                    tight_el[:, 1].eta, tight_el[:, 1].phi),
        False,
    )
    ev_res_EE = ev_res_EE_base & ev_res_EE_dr

    if ak.any(ev_res_EE):
        e1  = tight_el[ev_res_EE, 0]
        e2  = tight_el[ev_res_EE, 1]
        j1  = jet[ev_res_EE, 0]
        j2  = jet[ev_res_EE, 1]
        w   = weight[ev_res_EE]

        mll   = (e1 + e2).mass
        mlljj = (e1 + e2 + j1 + j2).mass

        dy_ee = (mll > 60)  & (mll < 150) & (mlljj > 800)   # DY CR
        sr_ee = (mll > 400) & (mlljj > 800)                  # SR

        fill_h(h, "Resolve_mll",   "DY_CR_EE", mll[dy_ee],     w[dy_ee])
        fill_h(h, "Resolve_mlljj", "DY_CR_EE", mlljj[dy_ee],   w[dy_ee])
        fill_h(h, "Resolve_mll",   "SR_EE",    mll[sr_ee],     w[sr_ee])
        fill_h(h, "Resolve_mlljj", "SR_EE",    mlljj[sr_ee],   w[sr_ee])
        fill_h(h, "Resolve_l1pt",  "SR_EE",    e1[sr_ee].pt,   w[sr_ee])
        fill_h(h, "Resolve_l2pt",  "SR_EE",    e2[sr_ee].pt,   w[sr_ee])
        fill_h(h, "Resolve_j1pt",  "SR_EE",    j1[sr_ee].pt,   w[sr_ee])
        fill_h(h, "Resolve_j2pt",  "SR_EE",    j2[sr_ee].pt,   w[sr_ee])
        fill_h(h, "Resolve_njet",  "SR_EE",
               ak.num(jet[ev_res_EE])[sr_ee].astype(float), w[sr_ee])

    # ── MM Resolved ─────────────────────────────────────────
    ev_res_MM_base = (
        is_MM & (n_tight == 2) & has2j &
        (tight_mu[:, 0].pt > 60.0) & (tight_mu[:, 1].pt > 53.0) &
        (tight_mu[:, 0].pt > cfg["mu_pt"]) & trig_mu
    )
    ev_res_MM_dr = ak.where(
        ev_res_MM_base,
        jet_dr_pass(tight_mu[:, 0].eta, tight_mu[:, 0].phi,
                    tight_mu[:, 1].eta, tight_mu[:, 1].phi),
        False,
    )
    ev_res_MM = ev_res_MM_base & ev_res_MM_dr

    if ak.any(ev_res_MM):
        m1  = tight_mu[ev_res_MM, 0]
        m2  = tight_mu[ev_res_MM, 1]
        j1  = jet[ev_res_MM, 0]
        j2  = jet[ev_res_MM, 1]
        w   = weight[ev_res_MM]

        mll   = (m1 + m2).mass
        mlljj = (m1 + m2 + j1 + j2).mass

        dy_mm = (mll > 60)  & (mll < 150) & (mlljj > 800)
        sr_mm = (mll > 400) & (mlljj > 800)

        # 뮤온 tight charge (SS 구분용)
        tc1 = tight_mu[ev_res_MM, 0].tightCharge
        tc2 = tight_mu[ev_res_MM, 1].tightCharge
        tight_charge = (tc1 == 2) & (tc2 == 2)

        fill_h(h, "Resolve_mll",   "DY_CR_MM",       mll[dy_mm],   w[dy_mm])
        fill_h(h, "Resolve_mlljj", "DY_CR_MM",       mlljj[dy_mm], w[dy_mm])
        fill_h(h, "Resolve_mll",   "SR_MM",          mll[sr_mm],   w[sr_mm])
        fill_h(h, "Resolve_mlljj", "SR_MM",          mlljj[sr_mm], w[sr_mm])
        fill_h(h, "Resolve_mll",   "SR_MM_SS_tight", mll[sr_mm & (m1.charge * m2.charge > 0) & tight_charge], w[sr_mm & (m1.charge * m2.charge > 0) & tight_charge])
        fill_h(h, "Resolve_mll",   "SR_MM_OS_tight", mll[sr_mm & (m1.charge * m2.charge < 0) & tight_charge], w[sr_mm & (m1.charge * m2.charge < 0) & tight_charge])

    # ── EM Resolved (Flavor CR) ──────────────────────────────
    # 리딩: 뮤온 (Trigger), 서브리딩: 전자
    ev_res_EM_base = (
        is_EM & (n_tight == 2) & has2j &
        (tight_mu[:, 0].pt > 60.0) & (tight_el[:, 0].pt > 53.0) &
        (tight_mu[:, 0].pt > cfg["mu_pt"]) & trig_mu
    )
    ev_res_EM_dr = ak.where(
        ev_res_EM_base,
        jet_dr_pass(tight_mu[:, 0].eta, tight_mu[:, 0].phi,
                    tight_el[:, 0].eta, tight_el[:, 0].phi),
        False,
    )
    ev_res_EM = ev_res_EM_base & ev_res_EM_dr

    if ak.any(ev_res_EM):
        ml  = tight_mu[ev_res_EM, 0]
        el_ = tight_el[ev_res_EM, 0]
        j1  = jet[ev_res_EM, 0]
        j2  = jet[ev_res_EM, 1]
        w   = weight[ev_res_EM]

        mll   = (ml + el_).mass
        mlljj = (ml + el_ + j1 + j2).mass
        flav  = (mll > 400) & (mlljj > 800)

        fill_h(h, "Resolve_mll",   "Flav_CR_EM", mll[flav],   w[flav])
        fill_h(h, "Resolve_mlljj", "Flav_CR_EM", mlljj[flav], w[flav])

    # ════════════════════════════════════════════════════════
    #  BOOSTED 선택 (not IsResolvedEvent)
    # ════════════════════════════════════════════════════════
    is_resolved = ev_res_EE | ev_res_MM | ev_res_EM

    for is_lead_el in [True, False]:
        # 리딩 렙톤 종류 결정
        trig_ev  = trig_el if is_lead_el else trig_mu
        lead_col = tight_el if is_lead_el else tight_mu
        sf_col   = loose_el if is_lead_el else loose_mu   # same-flavor loose
        of_col   = loose_mu if is_lead_el else loose_el   # opposite-flavor loose
        pt_safe  = cfg["el_pt"] if is_lead_el else cfg["mu_pt"]
        label    = "EE"  if is_lead_el else "MM"
        flabel   = "EMJ" if is_lead_el else "MEJ"

        boost_base = (
            ~is_resolved &
            (ak.num(lead_col) >= 1) &
            (lead_col[:, 0].pt > 60.0) &
            (lead_col[:, 0].pt > pt_safe) &
            trig_ev
        )
        if not ak.any(boost_base):
            continue

        lead_lep  = lead_col[boost_base, 0]
        sf_loose  = sf_col[boost_base]
        of_loose  = of_col[boost_base]
        sel_fj    = fj[boost_base]
        sel_fjlsf = fj_lsf[boost_base]
        w         = weight[boost_base]

        # ── BOOSTED DY CR: SF 느슨한 렙톤 60<mll<150 + back-to-back 팻젯 ──
        # 리딩과 같은 sf_loose 중에서 60 < mll < 150 짝 탐색
        if ak.any(ak.num(sf_loose) > 1):   # 리딩 제외 후 추가 렙톤 필요
            # 리딩과 sf_loose 각 원소 사이의 mll 계산
            mll_pairs = (lead_lep[:, None] + sf_loose).mass    # [event, n_sf]

            # 리딩 자신 제외: ΔR > 0.01
            dr_self = delta_r2(
                lead_lep.eta[:, None], lead_lep.phi[:, None],
                sf_loose.eta, sf_loose.phi,
            )
            valid_pairs = (dr_self > 0.0001) & (mll_pairs > 60) & (mll_pairs < 150)
            has_lowmll  = ak.any(valid_pairs, axis=1)

            if ak.any(has_lowmll & (ak.num(sel_fj) > 0)):
                dphi_fj = delta_phi_abs(
                    lead_lep.phi[:, None], sel_fj.phi
                )
                has_btb = ak.any(dphi_fj > 2.0, axis=1)
                dy_sel  = has_lowmll & has_btb

                if ak.any(dy_sel):
                    # 첫 번째 back-to-back 팻젯 사용
                    hn_fj   = ak.firsts(sel_fj[dy_sel][dphi_fj[dy_sel] > 2.0])
                    ll_dy   = lead_lep[dy_sel]
                    w_dy    = w[dy_sel]
                    mll_dy  = ak.firsts(mll_pairs[dy_sel][valid_pairs[dy_sel]])

                    wr_cand = ll_dy + hn_fj
                    mlljj   = wr_cand.mass
                    wr_ok   = mlljj > 800

                    fill_h(h, "Boost_mlljj", f"DY_CR_{label}", mlljj[wr_ok],       w_dy[wr_ok])
                    fill_h(h, "Boost_mll",   f"DY_CR_{label}", mll_dy[wr_ok],      w_dy[wr_ok])
                    fill_h(h, "Boost_l1pt",  f"DY_CR_{label}", ll_dy[wr_ok].pt,    w_dy[wr_ok])
                    fill_h(h, "Boost_fjpt",  f"DY_CR_{label}", hn_fj[wr_ok].pt,    w_dy[wr_ok])
                    fill_h(h, "Boost_fjsdm", f"DY_CR_{label}", hn_fj[wr_ok].msoftdrop, w_dy[wr_ok])
                    fill_h(h, "Boost_dphi",  f"DY_CR_{label}",
                           delta_phi_abs(ll_dy[wr_ok].phi, hn_fj[wr_ok].phi), w_dy[wr_ok])

        # ── BOOSTED SR / Flavor CR: mll > 150 (낮은 mll 렙톤 없음) ──
        # LSF fat jet + 팻젯 안쪽 느슨한 렙톤 탐색
        if not ak.any(ak.num(sel_fjlsf) > 0):
            continue

        dphi_lsf = delta_phi_abs(lead_lep.phi[:, None], sel_fjlsf.phi)
        has_btb_lsf = ak.any(dphi_lsf > 2.0, axis=1)
        if not ak.any(has_btb_lsf):
            continue

        hn_fj  = ak.firsts(sel_fjlsf[has_btb_lsf][dphi_lsf[has_btb_lsf] > 2.0])
        ll_boost = lead_lep[has_btb_lsf]
        sf_b   = sf_loose[has_btb_lsf]
        of_b   = of_loose[has_btb_lsf]
        w_b    = w[has_btb_lsf]

        wr_cand = ll_boost + hn_fj
        wr_m    = wr_cand.mass

        # SF/OF 느슨한 렙톤이 팻젯 안에 (ΔR < 0.8, pT > 53 GeV)
        def lep_in_fj(col):
            if not ak.any(ak.num(col) > 0):
                return ak.zeros_like(wr_m, dtype=bool), None
            pt_ok = col.pt > 53.0
            dr2_  = delta_r2(
                hn_fj.eta[:, None], hn_fj.phi[:, None],
                col.eta, col.phi,
            )
            in_fj = (dr2_ < 0.64) & pt_ok
            return ak.any(in_fj, axis=1), ak.firsts(col[in_fj])

        sf_in, sf_lep = lep_in_fj(sf_b)
        of_in, of_lep = lep_in_fj(of_b)

        # 추가 강한 렙톤 없음 (extra tight lepton veto)
        no_extra = ak.num(lead_col[boost_base][has_btb_lsf]) <= 1

        # ── Boosted SR: SF in FJ, no OF, mll > 200, WR > 800 ──
        if sf_lep is not None:
            mll_sf = (ll_boost + sf_lep).mass
            sr_sel = sf_in & ~of_in & no_extra & (mll_sf > 200) & (wr_m > 800)
            if ak.any(sr_sel):
                fill_h(h, "Boost_mlljj",  f"SR_{label}",   wr_m[sr_sel],            w_b[sr_sel])
                fill_h(h, "Boost_l1pt",   f"SR_{label}",   ll_boost[sr_sel].pt,     w_b[sr_sel])
                fill_h(h, "Boost_fjpt",   f"SR_{label}",   hn_fj[sr_sel].pt,        w_b[sr_sel])
                fill_h(h, "Boost_fjsdm",  f"SR_{label}",   hn_fj[sr_sel].msoftdrop, w_b[sr_sel])
                fill_h(h, "Boost_fjlsf3", f"SR_{label}",   hn_fj[sr_sel].lsf3,      w_b[sr_sel])
                fill_h(h, "Boost_dphi",   f"SR_{label}",
                       delta_phi_abs(ll_boost[sr_sel].phi, hn_fj[sr_sel].phi), w_b[sr_sel])

        # ── Boosted Flavor CR: OF in FJ, no SF, mll > 200, WR > 800 ──
        if of_lep is not None:
            mll_of = (ll_boost + of_lep).mass
            flav_sel = ~sf_in & of_in & no_extra & (mll_of > 200) & (wr_m > 800)
            if ak.any(flav_sel):
                fill_h(h, "Boost_mlljj",  f"Flav_{flabel}",  wr_m[flav_sel],            w_b[flav_sel])
                fill_h(h, "Boost_l1pt",   f"Flav_{flabel}",  ll_boost[flav_sel].pt,     w_b[flav_sel])
                fill_h(h, "Boost_fjpt",   f"Flav_{flabel}",  hn_fj[flav_sel].pt,        w_b[flav_sel])
                fill_h(h, "Boost_fjlsf3", f"Flav_{flabel}",  hn_fj[flav_sel].lsf3,      w_b[flav_sel])


# ══════════════════════════════════════════════════════════════
#  파일 루프 및 히스토그램 저장
# ══════════════════════════════════════════════════════════════

def run(input_files: List[str], era: str, is_data: bool,
        output: str = "wr_histos.root") -> None:
    """NanoAOD 파일 목록을 처리하고 히스토그램을 ROOT 파일로 저장"""
    histos = book_histograms()

    for fname in input_files:
        print(f"처리 중: {fname}")
        events = NanoEventsFactory.from_root(
            fname, schemaclass=NanoAODSchema, uproot_options={"timeout": 300}
        ).events()
        analyze_events(events, era=era, is_data=is_data, h=histos)

    with uproot.recreate(output) as f:
        for name, histo in histos.items():
            f[name] = histo
    print(f"히스토그램 저장 완료: {output}")


# ══════════════════════════════════════════════════════════════
#  엔트리 포인트
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="WR/LRSM 분석 (uproot + awkward-array)"
    )
    parser.add_argument("--input",   nargs="+", required=True,
                        help="입력 NanoAOD ROOT 파일")
    parser.add_argument("--era",     default="2022",
                        choices=list(ERA_CFG),
                        help="데이터 시기 (2017/2022/2022EE/2023/2023BPix)")
    parser.add_argument("--is-data", action="store_true",
                        help="실제 데이터 플래그 (기본값: MC)")
    parser.add_argument("--output",  default="wr_histos.root",
                        help="출력 ROOT 파일")
    args = parser.parse_args()

    run(
        input_files = args.input,
        era         = args.era,
        is_data     = args.is_data,
        output      = args.output,
    )

usage: ipykernel_launcher.py [-h] --input INPUT [INPUT ...]
                             [--era {2017,2022,2022EE,2023,2023BPix}]
                             [--is-data] [--output OUTPUT]
ipykernel_launcher.py: error: the following arguments are required: --input


SystemExit: 2

/home/achihwan/miniconda3/envs/hep-py-env/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
